In [ ]:
from common import *

## 11. Obrada nedostajucih podataka

### 11.1 Potpuno nedostajuci atributi na lokaciji

U prethodnom delu analize detektovali smo da postoje merne stanice koje nisu bile opremljene adekvatnim senzorima za merenje svih parametara. Zbog toga postoje kolone koje su za određene lokacije u potpunosti prazne. Ova situacija može ugroziti i otežati radi svih modela imputacije i predikcije koje ćemo u budućnosti konstruisati.

Utvrdimo i prikažimo koje to lokacije imaju ovakav tip anomalije i detektujmo o kojim se promenljivima radi.

In [ ]:
data = loadData("backups/weatherAusAfter9_2_6.csv")
def ispisiNpostojecePodatke():
    locations = data['Location'].unique()

    lokacije_sa_praznim_kolonama = []
    nedostajuci_atributi = tuple()
    prazniElementiDict = {}
    for loc in locations:
        subset = data[data['Location'] == loc]
        prazne_kolone = [col for col in data.columns if subset[col].isna().all()]
        
        if prazne_kolone:
            nedostajuci_atributi += tuple(prazne_kolone)
            lokacije_sa_praznim_kolonama.append(loc)
            print(f"Lokacija: {loc} ima prazne kolone: {prazne_kolone}")
            prazniElementiDict[loc] = prazne_kolone
    print("\nLista lokacija sa potpuno praznim kolonama za neke atribute:")
    print(lokacije_sa_praznim_kolonama)
    print("\nPotpuno nedostajući atributi u pojedinim meteorološkim stanicama:")
    print(set(nedostajuci_atributi))
    return lokacije_sa_praznim_kolonama, set(nedostajuci_atributi), prazniElementiDict

lokacije_sa_praznim_kolonama, nedostajuci_atributi, prazniElementiDict = ispisiNpostojecePodatke()

print(prazniElementiDict)


#### 11.1.1 Grupisanje mernih stanica na osnovu razdaljine

Jedna od mogućnosti kako bi se ovaj problem mogao rešiti je oslanjanje na podatke sa susednih mernih stanica. Takav pristup ima uporište u Toblerovom prvom zakonu geografije koji glasi "Sve je povezano sa svim ostalim, ali su bliske stvari povezanije od udaljenih stvari." 

Za primenu ovakvog pristupa neophodno je nejpre izračunati međusobnu idaljenosti svakog para meteoroloških stanica.

In [ ]:
def haversine_razdaljina(lat1, lon1, lat2, lon2):
    """Računa vazdušnu razdaljinu između dve tačke u kilometrima."""
    R = 6371.0 # Poluprečnik Zemlje u km
    
    lat1_rad, lon1_rad = math.radians(lat1), math.radians(lon1)
    lat2_rad, lon2_rad = math.radians(lat2), math.radians(lon2)

    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad

    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

def generisi_razdaljine(ulazni_fajl, izlazni_fajl):
    stanice = []
    
    with open(ulazni_fajl, mode='r', encoding='utf-8') as fajl:
        citac = csv.DictReader(fajl)
        for red in citac:
            stanice.append({
                'ime': red['StationName'],
                'lat': float(red['Lat']),
                'lon': float(red['Lon'])
            })
    print(f"Uspešno učitano {len(stanice)} stanica iz fajla '{ulazni_fajl}'.")
    brojac_parova = 0
    with open(izlazni_fajl, mode='w', encoding='utf-8', newline='') as izlaz:
        polja = ['stanica_1', 'stanica_2', 'razdaljina_km']
        pisac = csv.DictWriter(izlaz, fieldnames=polja)
        pisac.writeheader()
        for stanica1, stanica2 in combinations(stanice, 2):
            razdaljina = haversine_razdaljina(
                stanica1['lat'], stanica1['lon'], 
                stanica2['lat'], stanica2['lon']
            )
            pisac.writerow({
                'stanica_1': stanica1['ime'],
                'stanica_2': stanica2['ime'],
                'razdaljina_km': round(razdaljina, 2)
            })
            brojac_parova += 1
            
    print(f"Završeno! Izračunato i upisano {brojac_parova} razdaljina u fajl '{izlazni_fajl}'.")

ulazni_csv = 'GeoPodaci/stanice.csv'
izlazni_csv = 'GeoPodaci/udaljenost_stanica.csv'

generisi_razdaljine(ulazni_csv, izlazni_csv)

Naredni korak je kreiranje skupova bliskih lokacija - lokacija kod kojih međusobna udaljenost svaka dva člana skupa nije veća od prethodno definisanog praga. Za potrebe dalje analize kreiraćemo više pragova razdaljina.

In [ ]:
def pronadji_grupe_lokacija(ime_fajla, pragovi):
    df = pd.read_csv(ime_fajla)
    rezultati = {}
    
    for i in range(1,len(pragovi)):
        pragMin = pragovi[i-1]
        prag = pragovi[i]
        G = nx.Graph()
        for _, red in df.iterrows():
            lok1 = red['stanica_1']
            lok2 = red['stanica_2']
            udaljenost = red['razdaljina_km']
            
            if udaljenost <= prag and udaljenost > pragMin:
                G.add_edge(lok1, lok2)
        
        sve_grupe = list(nx.find_cliques(G))
        
        filtrirane_grupe = [grupa for grupa in sve_grupe if len(grupa) > 1]
        filtrirane_grupe.sort(key=len, reverse=True)
        rezultati[prag] = filtrirane_grupe
        
    return rezultati

trazeni_pragovi = [0,50, 100, 200, 300]
fajl = "GeoPodaci/udaljenost_stanica.csv"
izlazni_fajl = "StationClasters/pronadjene_grupe.csv"

pronadjene_grupe = pronadji_grupe_lokacija(fajl, trazeni_pragovi)

podaci_za_csv = []

if pronadjene_grupe:
    for prag in trazeni_pragovi:
        
        grupe_za_prag = pronadjene_grupe.get(prag, [])
        if grupe_za_prag:
            for i, grupa in enumerate(grupe_za_prag, 1):
                print(f"Grupa {i} ({len(grupa)} članova): {', '.join(grupa)}")
                
                podaci_za_csv.append({
                    'Prag_km': prag,
                    'Grupa_ID': i,
                    'Broj_clanova': len(grupa),
                    'Clanovi': ', '.join(grupa)
                })
        else:
            print("Nema pronađenih grupa za ovaj prag.")

if podaci_za_csv:
    df_izlaz = pd.DataFrame(podaci_za_csv)
    df_izlaz.to_csv(izlazni_fajl, index=False, encoding='utf-8')
    print(f"\nUspesno sačuvano u: {izlazni_fajl}")
else:
    print("\nNema podataka za čuvanje u CSV.")

#### 11.1.2 Racunanje matrice korelacija za svaku od grupa

Sada ćemo kreirati pomoćnu funkciju koja će za prosledjenu listu lokacija i određeni atribut kreirati matricu korelacije. Posto nam je bitna samo generalna statistika, kreriaćemo još jednu metodu koja će nam vratiti srednju vrednost korelacije.

In [ ]:
def izracunaj_korelaciju(lokacije, parametar):
    df_filtrirano = data[data['Location'].isin(lokacije)]
    pivot_tabela = df_filtrirano.pivot(
        index='Date', 
        columns='Location', 
        values=parametar
    )
    korelacija = pivot_tabela.corr()

def summarise_correlation(grupe, atributi):
    rezultati = {}
    
    for atribut in atributi:
        sve_korelacije = []
        
        for grupa in grupe:
            df_filtrirano = data[data['Location'].isin(grupa)]
            pivot = df_filtrirano.pivot(index='Date', columns='Location', values=atribut)
            
            corr_matrix = pivot.corr().abs()
            
            np.fill_diagonal(corr_matrix.values, np.nan)
            
            mean_corr = corr_matrix.stack().mean(skipna=True)
            
            if mean_corr is not None and not np.isnan(mean_corr) and mean_corr >= 0.0:
                sve_korelacije.append(mean_corr)
        
        if sve_korelacije:
            rezultati[atribut] = float(np.mean(sve_korelacije))
        else:
            rezultati[atribut] = None
    
    return rezultati


Ovu funkciju sada možemo iskoristiti i proveriti za koji od problematičnih atributa važi zakonitost. Otpočnimo sa najnižim pragom od 50km. Takodje, trenutno u analizi cemo koristiti iskljucivo numericke promenljive.

In [ ]:
def generisi_matricu_korelacija_po_grupama_i_atributima(data, razdaljina, nedostajuci_atributi, pronadjene_grupe):
    data = data[data['Date'] < SPLIT_DATE]
    numericki_atributi = [x for x in set(nedostajuci_atributi) if x in data.select_dtypes(include=[np.number]).columns.tolist()]
    grupe = pronadjene_grupe.get(razdaljina, [])
    if not grupe:
        print(f"  Nema pronađenih grupa za razdaljinu {razdaljina} km. Preskačem...")
        return {}
    podaci_za_matricu = []


    rezultati = {}
    for i, grupa in enumerate(grupe):
        rezultati[";".join(grupa)] = {}
        ime_grupe = f"Grupa {i+1} ({len(grupa)} lok.)"
        red_podaci = {'Grupa': ime_grupe}
        
        df_grupa = data[data['Location'].isin(grupa)]
        
        # Računamo korelaciju za svaki numerički atribut
        for atribut in numericki_atributi:
            pivot = df_grupa.pivot(index='Date', columns='Location', values=atribut)
            corr_matrix = pivot.corr().abs()
            
            # Postavljamo glavnu dijagonalu na NaN kako ne bi veštački podigla prosek
            np.fill_diagonal(corr_matrix.values, np.nan)
            
            # Prosečna korelacija matrice (sada bez glavne dijagonale)
            mean_corr = corr_matrix.stack().mean(skipna=True)
            
            if pd.notna(mean_corr) and mean_corr >= 0.0:
                red_podaci[atribut] = mean_corr
                rezultati[";".join(grupa)][atribut] = mean_corr
            else:
                red_podaci[atribut] = np.nan
                rezultati[";".join(grupa)][atribut] = np.nan
        podaci_za_matricu.append(red_podaci)

    df_matrica = pd.DataFrame(podaci_za_matricu)
    df_matrica.set_index('Grupa', inplace=True)

    naziv_fajla = f'reports/claster_corelations/korelacije_po_grupama_i_atributima_razdljina_{razdaljina}.csv'
    df_matrica.to_csv(naziv_fajla)
    print(f"Matrica uspešno sačuvana kao '{naziv_fajla}'.\n")

    plt.figure(figsize=(10, 6))
    sns.heatmap(
        df_matrica, 
        annot=True,    
        cmap='YlGnBu', 
        fmt=".3f",     
        linewidths=0.5,
        cbar_kws={'label': 'Prosečna apsolutna korelacija'},
        vmin=df_matrica.min().min(), 
        vmax=1.0,
    )

    plt.title(f'Toplotni grafik korelacija po grupama i atributima razdaljina {razdaljina} km', fontsize=15, pad=20)
    plt.xlabel('Atributi', fontsize=12)
    plt.ylabel('Grupe lokacija', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    plt.show()
    return rezultati
kor = {}
for razdaljina in trazeni_pragovi:
    kor[str(razdaljina)] = generisi_matricu_korelacija_po_grupama_i_atributima(data, razdaljina, nedostajuci_atributi, pronadjene_grupe)


Primećujemo da prosečna korelacija između stanica u grupi dosledno opada sa povećanjem radijusa grupisanja: sa oko 0.89 (medijana 0.98) na 50km, preko 0.82 na 100km i 0.75 na 200km, do svega 0.66 (medijana 0.67) na 300km - očekivano, jer bliže stanice dele isti sinoptički sistem, dok udaljenije sve više odražavaju lokalnu klimu.

Efekat pritom nije isti za sve atribute. `Pressure9am`, `Pressure3pm` (i njihove lag/dnevna-razlika varijante) i `Sunshine` ostaju izuzetno korelisani (>0.93) čak i na 300km, jer zavise od sinoptičkih sistema koji se prostiru na stotinama kilometara. Nasuprot tome, `Cloud9am` i `Cloud3pm` opadaju mnogo brže: već na najmanjem radijusu od 50km nijedna od 4 dostupne grupe ne dostiže korelaciju iznad 0.99 (maksimum je 0.982), dok kod `Pressure9am` sve 4 dostupne grupe prelaze taj prag. Ovo objašnjava zašto će u narednom odeljku (11.1.4) prostorna imputacija sa pragom korelacije >0.99 praktično uvek uspevati za `Pressure`/`Sunshine`, a retko ili nikad za `Cloud` atribute - za njih ćemo se osloniti prevashodno na Random Forest imputaciju (odeljak 11.2.2).

#### 11.1.3 Odabir modela za imputaciju

In [ ]:
atributi_za_analizu = ['Evaporation', 'Sunshine', "Pressure9am", "Pressure3pm", "Cloud3pm", "Cloud9am"]
prazniElementiDictFiltered = {k:[a for a in v if a in atributi_za_analizu] for k,v in prazniElementiDict.items() if any(attr in v for attr in atributi_za_analizu)}
print(prazniElementiDictFiltered)

In [ ]:
FAJL_UDALJENOSTI = 'GeoPodaci/udaljenost_stanica.csv'

df = data.copy()
df_dist = pd.read_csv(FAJL_UDALJENOSTI)

df['Lon'] = pd.to_numeric(df['Lon'], errors='coerce')
df['Lat'] = pd.to_numeric(df['Lat'], errors='coerce')

# Rečnik za praćenje svih rangova kroz sve grupe kako bismo na kraju izvukli prosek
istorija_rangova = {}

# Definisanje pondera za finalni skor (možeš ih prilagoditi po želji)
# Zbir pondera bi idealno trebalo da bude 1.0
PONDERI = {
    'MAE': 0.40,  # Fokus na prosečnu apsolutnu grešku
    'RMSE': 0.40, # Fokus na kažnjavanje velikih odstupanja (outliera)
    'R2': 0.20    # Manja težina na koeficijent determinacije zbog manjih uzoraka
}

with open("reports/rezultati_analize_imputacionih_modela.txt", "w", encoding="utf-8") as f:
    
    for nedostajuci_atribut in atributi_za_analizu:
        CILJANI_ATRIBUT = nedostajuci_atribut
        lokacijeKojimaFaliAtribut = [lokacija for lokacija, atributi in prazniElementiDictFiltered.items() if CILJANI_ATRIBUT in atributi]

        istorija_rangova[CILJANI_ATRIBUT] = {
            'IDW': [], 'KNN': [], 'LinearRegression': [], 'RandomForest': [], 'XGBoost': []
        }

        for lokacije in pronadjene_grupe.get(50, []):
            if any(lokacija in lokacijeKojimaFaliAtribut for lokacija in lokacije):
                continue    
            
            MASKIRANA_STANICA = None
            for potencijalna_stanica in lokacije:
                if potencijalna_stanica in lokacijeKojimaFaliAtribut:
                    continue
                    
                if df[df['Location'] == potencijalna_stanica][CILJANI_ATRIBUT].count() > 10:
                    MASKIRANA_STANICA = potencijalna_stanica
                    break

            if MASKIRANA_STANICA is None:
                continue

            df_local = df[df['Location'].isin(lokacije)].copy()

            udaljenosti_dict = {}
            for _, row in df_dist.iterrows():
                par = tuple(sorted([row['stanica_1'], row['stanica_2']]))
                udaljenosti_dict[par] = row['razdaljina_km']

            maska_test_stanice = df_local['Location'] == MASKIRANA_STANICA
            df_original = df_local[maska_test_stanice][['Date', CILJANI_ATRIBUT]].copy()
            df_original.rename(columns={CILJANI_ATRIBUT: 'Stvarna_Vrednost'}, inplace=True)

            df_local.loc[maska_test_stanice, CILJANI_ATRIBUT] = np.nan

            donor_stanice = [s for s in lokacije if s != MASKIRANA_STANICA]
            df_donori = df_local[df_local['Location'].isin(donor_stanice)]

            dnevni_prosek = df_donori.groupby('Date')[CILJANI_ATRIBUT].mean().reset_index()
            dnevni_prosek.rename(columns={CILJANI_ATRIBUT: 'Prosek_Donora'}, inplace=True)

            df_local = df_local.merge(dnevni_prosek, on='Date', how='left')

            features = ['Lat', 'Lon', 'Nadmorska visina (m)', 'Prosek_Donora']

            df_train = df_local[df_local['Location'] != MASKIRANA_STANICA].dropna(subset=features + [CILJANI_ATRIBUT])

            X_train = df_train[features]
            y_train = df_train[CILJANI_ATRIBUT]

            df_test = df_local[df_local['Location'] == MASKIRANA_STANICA]
            df_test_clean = df_test.dropna(subset=features).copy() 
                
            X_test = df_test_clean[features]

            rezultati_predikcija = pd.merge(df_test_clean[['Date']], df_original, on='Date', how='left')

            idw_predikcije = []
            for index, row in df_test_clean.iterrows():
                datum = row['Date']
                donori_danas = df_train[df_train['Date'] == datum]
                
                if donori_danas.empty:
                    idw_predikcije.append(np.nan)
                    continue
                    
                tezine = []
                vrednosti = []
                for _, donor_row in donori_danas.iterrows():
                    donor_ime = donor_row['Location']
                    par = tuple(sorted([MASKIRANA_STANICA, donor_ime]))
                    dist = udaljenosti_dict.get(par, np.nan)
                    
                    if pd.notna(dist) and dist > 0:
                        tezine.append(1.0 / dist)
                        vrednosti.append(donor_row[CILJANI_ATRIBUT])
                        
                if np.sum(tezine) > 0:
                    idw_predikcije.append(np.average(vrednosti, weights=tezine))
                else:
                    idw_predikcije.append(np.nan)

            rezultati_predikcija['IDW'] = idw_predikcije

            if len(X_train) > 0:
                broj_komsija = max(1, min(4, len(X_train))) 
                knn = KNeighborsRegressor(n_neighbors=broj_komsija, weights='distance')
                knn.fit(X_train, y_train)
                rezultati_predikcija['KNN'] = knn.predict(X_test)

                lr = LinearRegression()
                lr.fit(X_train, y_train)
                rezultati_predikcija['LinearRegression'] = lr.predict(X_test)
            
                rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
                rf.fit(X_train, y_train)
                rezultati_predikcija['RandomForest'] = rf.predict(X_test)

                xgboost = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
                xgboost.fit(X_train, y_train)
                rezultati_predikcija['XGBoost'] = xgboost.predict(X_test)
            else:
                print(f"Nema dovoljno podataka za trening ML modela (stanica: {MASKIRANA_STANICA})", file=f)
                continue

            print("\n" + "="*100, file=f)
            print(f"Evaluacija za atribut: {nedostajuci_atribut}", file=f)
            print(f"Evaluacija grupe: {lokacije}", file=f)
            print(f"Maskirana stanica: {MASKIRANA_STANICA}", file=f)
            print("="*100, file=f)
            
            donor_stanice_info = []
            for donor in donor_stanice:
                par = tuple(sorted([MASKIRANA_STANICA, donor]))
                dist = udaljenosti_dict.get(par, None)
                if pd.notna(dist):
                    donor_stanice_info.append(f"{donor} ({dist:.1f} km)")
                else:
                    donor_stanice_info.append(f"{donor} (N/A km)")

            print(f"Donor lokacije u grupi: {', '.join(donor_stanice_info)}", file=f)
            print("Atributi na osnovu kojih se vrši predikcija: " + ", ".join(features), file=f)
            print("-" * 100, file=f)

            modeli = ['IDW', 'KNN', 'LinearRegression', 'RandomForest', 'XGBoost']
            evaluacija = []

            rez_clean = rezultati_predikcija.dropna()
            y_true = rez_clean['Stvarna_Vrednost']

            if rez_clean.empty:
                print(f"Ne može se izračunati evaluacija, premalo preklapanja podataka.", file=f)
                continue

            for model in modeli:
                y_pred = rez_clean[model]
                mae = mean_absolute_error(y_true, y_pred)
                rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                r2 = r2_score(y_true, y_pred)
                
                evaluacija.append({
                    'Model': model,
                    'MAE': mae,
                    'RMSE': rmse,
                    'R^2': r2
                })

            df_eval = pd.DataFrame(evaluacija)

            df_eval['MAE_inv'] = -df_eval['MAE']
            df_eval['RMSE_inv'] = -df_eval['RMSE']
            df_eval['R2_inv'] = df_eval['R^2'] # R^2 je već u formatu "više je bolje"
            
            def min_max_scale(series):
                min_val = series.min()
                max_val = series.max()
                if max_val == min_val:
                    return pd.Series(1.0, index=series.index)
                return (series - min_val) / (max_val - min_val)
            
            df_eval['MAE_norm'] = min_max_scale(df_eval['MAE_inv'])
            df_eval['RMSE_norm'] = min_max_scale(df_eval['RMSE_inv'])
            df_eval['R2_norm'] = min_max_scale(df_eval['R2_inv'])
            
            df_eval['Score'] = (
                (df_eval['MAE_norm'] * PONDERI['MAE']) + 
                (df_eval['RMSE_norm'] * PONDERI['RMSE']) + 
                (df_eval['R2_norm'] * PONDERI['R2'])
            )
            
            df_eval['Rang'] = df_eval['Score'].rank(ascending=False, method='min').astype(int)

            for _, row in df_eval.iterrows():
                istorija_rangova[nedostajuci_atribut][row['Model']].append(row['Rang'])

            df_eval['MAE'] = df_eval['MAE'].round(4)
            df_eval['RMSE'] = df_eval['RMSE'].round(4)
            df_eval['R^2'] = df_eval['R^2'].round(4)
            df_eval['Score'] = df_eval['Score'].round(4)
            
            kolone_za_brisanje = ['MAE_inv', 'RMSE_inv', 'R2_inv', 'MAE_norm', 'RMSE_norm', 'R2_norm']
            df_eval = df_eval.drop(columns=kolone_za_brisanje)

            print(df_eval.to_string(), file=f)
            print("="*100, file=f)

    print("\n\n" + "#"*100, file=f)
    print("Finalni izveštaj: Prosečan rang modela po predviđanim atributima", file=f)
    print("#"*100, file=f)
    
    for atribut, modeli_dict in istorija_rangova.items():
        print(f"\nAnaliza za atribut: {atribut}", file=f)
        prosecni_rangovi = {}
        
        for model_ime, lista_rangova in modeli_dict.items():
            if len(lista_rangova) > 0:
                prosecni_rangovi[model_ime] = sum(lista_rangova) / len(lista_rangova)
            else:
                prosecni_rangovi[model_ime] = None
                
        prosecni_sortirani = sorted(
            [(m, r) for m, r in prosecni_rangovi.items() if r is not None], 
            key=lambda item: item[1]
        )
        
        if not prosecni_sortirani:
            print("  Nema dovoljno uspešnih predikcija za statistiku.", file=f)
        else:
            for i, (model, prosecan_rang) in enumerate(prosecni_sortirani, 1):
                print(f"  {i}. {model.ljust(18)} - Prosečan rang: {prosecan_rang:.2f}", file=f)
        print("-" * 50, file=f)

#### 11.1.4 Imputacija

Naredni kod vrši automatizovanu imputaciju potpuno nedostajućih podataka za pojedinačne lokacije, oslanjajući se na informacije iz njima srodnih meteoroloških stanica. Za svaki atribut čija merenja u potpunosti nedostaju na određenoj stanici, algoritam analizira podatke iz njene grupe i strogo proverava da li postoji visoka međusobna korelacija (iznad 0,99) za taj specifični parametar. Ukoliko je ovaj uslov pouzdanosti ispunjen, izračunava se prosečna dnevna vrednost na nivou cele grupe, koja se zatim ciljano upisuje samo na mesta gde podaci ne postoje.

In [ ]:
old_df = data.copy()
def popuni_po_grupama(df, grupe, atribut, min_corr, korelacije):
    df_copy = df.copy()
    

    uspesanXGBoost = 0
    neuspesanXGBoost = 0
    kljucevi = korelacije.keys() if korelacije else []
    
    udaljenosti_dict = {}
    if atribut not in ['Cloud3pm', 'Cloud9am']:
        df_dist = pd.read_csv('GeoPodaci/udaljenost_stanica.csv')
            
        for _, row in df_dist.iterrows():
            par = tuple(sorted([row['stanica_1'], row['stanica_2']]))
            udaljenosti_dict[par] = row['razdaljina_km']
    
    for grupa in grupe:
        
        df_local = df_copy[df_copy['Location'].isin(grupa)]
        if df_local.empty:
            continue
            
        for loc in grupa:
            maska_lokacija = (df_copy['Location'] == loc)
            ukupno_merenja = maska_lokacija.sum()
            
            maska_nedostajuci = maska_lokacija & (df_copy[atribut].isna())
            nedostaje_merenja = maska_nedostajuci.sum()
            
            if nedostaje_merenja > 0 and nedostaje_merenja == ukupno_merenja:
                print(f"Lokaciji {loc} nedostaju svi podaci za atribut {atribut}. Trazim odgovarajuce grupe")
                odgovrajacuciKljucevi = [k for k in kljucevi if loc in k.split(';')]
                if not odgovrajacuciKljucevi:
                    print(f"Nema odgovarajućih grupa za lokaciju {loc} i atribut {atribut}. Preskačem.\n")
                    continue
                else:
                    validni_kljucevi = []
                    print("Pronadjene su sledece grupe sa navedenim korelacijama:")
                    for k in odgovrajacuciKljucevi:
                        print(f"  Grupa: {k.split(';')} - Prosečna korelacija: {korelacije[k].get(atribut, 0):.4f}")
                        if korelacije[k].get(atribut, 0) is not None and not np.isnan(korelacije[k].get(atribut, 0)):
                            validni_kljucevi.append(k)
                    if not validni_kljucevi:
                        print("Sve dostupne grupe imaju NaN ili None korelacije. Preskačem.\n")
                        continue
                    najbolja_grupa_kljuc = max(validni_kljucevi, key=lambda k: korelacije[k].get(atribut, 0))
                    if korelacije[najbolja_grupa_kljuc].get(atribut, 0) is None or np.isnan(korelacije[najbolja_grupa_kljuc].get(atribut, 0)):
                        print(f"Prosečna korelacija za grupu {najbolja_grupa_kljuc.split(';')} je NaN. Preskačem.\n")
                        continue
                    grupa = najbolja_grupa_kljuc.split(';')
                    print(f"Odabrao sam grupu: {grupa} sa prosečnom korelacijom {korelacije[najbolja_grupa_kljuc].get(atribut, 0):.4f}")
                    if korelacije[najbolja_grupa_kljuc].get(atribut, 0) < min_corr:
                        print(f"Prosečna korelacija {korelacije[najbolja_grupa_kljuc].get(atribut, 0):.4f} je ispod praga {min_corr}. Preskačem.\n")
                        continue
                
                donori = [d for d in grupa if d != loc]
                if not donori:
                    continue
                
                if atribut in ['Cloud3pm', 'Cloud9am']:
                    df_donori = df_copy[df_copy['Location'].isin(donori)][['Date', atribut]]
                    dnevni_prosek = df_donori.groupby('Date')[atribut].mean().reset_index()
                    dnevni_prosek.rename(columns={atribut: 'Prosek_Donora'}, inplace=True)
                    
                    temp_df = df_local.merge(dnevni_prosek, on='Date', how='left')
                    
                    features = ['Lat', 'Lon', 'Nadmorska visina (m)', 'Prosek_Donora']
                    
                    df_train = temp_df[(temp_df['Location'] != loc) & (temp_df['Date'] < SPLIT_DATE)].dropna(subset=features + [atribut])
                    
                    df_pred = temp_df[(temp_df['Location'] == loc) & (temp_df[atribut].isna())]
                    
                    if len(df_train) > 10 and not df_pred.empty:
                        X_train = df_train[features]
                        y_train = df_train[atribut]
                        X_pred = df_pred[features]
                        
                        model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, n_jobs=-1)
                        model.fit(X_train, y_train)
                        preds = model.predict(X_pred)
                        
                        datumi_za_upis = df_pred['Date'].values
                        df_copy.loc[(df_copy['Location'] == loc) & (df_copy['Date'].isin(datumi_za_upis)), atribut] = preds
                        
                        print(f"  -> Primenjen XGBoost za {loc}\n")
                        uspesanXGBoost += 1
                    else:
                        print(f"  -> Nema dovoljno validnih redova za XGBoost obuku (stanica: {loc})\n")
                        neuspesanXGBoost += 1
                        
                else:
                    podaci_idw = df_local[['Date', 'Location', atribut]]
                    pivot = podaci_idw.pivot(index='Date', columns='Location', values=atribut)
                    
                    if loc not in pivot.columns:
                        continue
                        
                    datumi_fale = df_copy[maska_nedostajuci]['Date']
                    
                    for datum in datumi_fale:
                        if datum not in pivot.index:
                            continue
                            
                        row_data = pivot.loc[datum]
                        donori_dostupni = [d for d in donori if d in row_data.index and pd.notna(row_data[d])]
                        
                        if not donori_dostupni:
                            continue
                            
                        tezine = []
                        vrednosti = []
                        for donor in donori_dostupni:
                            par = tuple(sorted([loc, donor]))
                            dist = udaljenosti_dict.get(par, np.nan)
                            
                            if pd.notna(dist) and dist > 0:
                                tezine.append(1.0 / dist)
                                vrednosti.append(row_data[donor])
                                
                        if tezine and np.sum(tezine) > 0:
                            procenjena_vrednost = np.average(vrednosti, weights=tezine)
                            # Upis precizno u red za tu lokaciju i taj datum
                            df_copy.loc[(df_copy['Location'] == loc) & (df_copy['Date'] == datum), atribut] = procenjena_vrednost
                    
                    print(f"  -> Primenjen IDW za {loc}\n")

    print("\nUspesan XGBoost popunjavanja:", uspesanXGBoost)
    print("Neuspesan XGBoost popunjavanja:", neuspesanXGBoost)
    
    return df_copy
for atribut in atributi_za_analizu:
    nedostajuci_count_pocetak = data[atribut].isna().sum()
    print(f"\n{'='*40}")
    print(f"--- Obrada atributa: {atribut} ---")
    print(f"{'='*40}")
    
    # 1. Prvi prolaz - pokušaj popunjavanja sa grupama do 50 km
    print("[50 km] Započinjem popunjavanje...")
    data = popuni_po_grupama(
        data,  
        pronadjene_grupe.get(50, []), 
        atribut, 
        0.95, 
        kor.get("50", {})
    )
    
    preostalo_nedostajucih = data[atribut].isna().sum()
    
    if preostalo_nedostajucih > 0:
        print(f"\n[info] Ostalo je {preostalo_nedostajucih} nedostajućih vrednosti za {atribut}.")
        print("[100 km] Pokrećem prošireno popunjavanje...")
        data = popuni_po_grupama(
            data, 
            pronadjene_grupe.get(100, []), 
            atribut, 
            0.95, 
            kor.get("100", {})
        )
        
        # Opciono: Ispis koliko je ostalo nakon 100 km
        konacno_nedostaje = data[atribut].isna().sum()
        print(f"\nNa pocetku bilo je {nedostajuci_count_pocetak} nedostajućih vrednosti za {atribut}.")
        print(f"Nakon 50km bilo je {preostalo_nedostajucih} nedostajućih vrednosti za {atribut}.")
        if konacno_nedostaje > 0:
            print(f"I nakon 100 km ostalo je {konacno_nedostaje} nedostajućih vrednosti.")
        else:
            print(f"Sve preostale vrednosti su uspešno popunjene sa 100 km.")
            
    else:
        print(f"\nSve vrednosti za {atribut} su uspešno popunjene u krugu od 50 km.")
    write_log(data, "Popunjen atribut " + atribut, None)
print("Imputacija je gotova, kreiramo grafike da proverimo koliko se podataka popunilo")
data = createVaribales(data)
write_log(data, "Imputirani svi potpuno nedostajuci atributi", "weatherAusAfter11_1_4.csv")
data = kreirajAnalizu(data)